# Web Search on Amazon Bedrock AgentCore for LangGraph developers

[Web Search on Amazon Bedrock AgentCore](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-connector-web-search-tool.html)
is a managed web search tool backed by an index that Amazon operates. This notebook goes
through what it offers a LangGraph agent, one feature at a time, and every claim is
checked against a live search rather than asserted:

1. **Search with no API key.** Calls are signed with SigV4 from your AWS credentials, and
   access is an IAM grant. Queries are served inside AWS and are not sent to a
   third-party search engine.
2. **What a result contains.** Web results carry a title, URL, publication date and an
   extract. Some results are structured facts rather than page extracts.
3. **Filters the model chooses.** Domain include and exclude lists and a published-date
   window, set per call.
4. **Policy the model cannot change.** A domain blocklist set by an administrator on the
   gateway target, which no tool call can lift.
5. **Per-tenant policy set by your code.** Blocklists taken from LangGraph runtime context,
   which never appear in the model's tool schema.
6. **Citations as data.** Results returned as a tool artifact, so your graph can check
   that every URL in the answer came from a search.
7. **Errors the model can correct, and multi-turn threads.**

If you only want the shortest path to a working agent, start with
[`agentcore_web_search_agent.ipynb`](./agentcore_web_search_agent.ipynb).

## Prerequisites

- An AWS account with Amazon Bedrock AgentCore access, working in `us-east-1`, `eu-west-1`
  or `ap-northeast-1`, the Regions that offer the web search connector. Outside those Regions,
  creating a target fails with "Connector integration web-search is not available for this
  account", the same message you get when the connector isn't enabled for the account, so
  check the Region first.
- AWS credentials configured as environment variables (`AWS_ACCESS_KEY_ID`,
  `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`) or through an AWS CLI profile.
- Python 3.10 or later.
- Model access in Amazon Bedrock for the Claude model used below. Any Bedrock chat model that
  supports tool use works.
- IAM permissions for your identity to create, read and delete AgentCore gateways and gateway
  targets, `bedrock-agentcore:InvokeGateway` on the gateway, and `bedrock:InvokeModel`. If you let
  the notebook create the gateway's service role in Step 1, you also need `iam:GetRole`,
  `iam:CreateRole`, `iam:PutRolePolicy` and `iam:PassRole`.

## Cost

Web Search is priced at $7 per 1,000 queries, and AgentCore Gateway invocations are
billed separately. A full run of this notebook makes a few dozen searches plus the
model calls. See [Amazon Bedrock AgentCore pricing](https://aws.amazon.com/bedrock/agentcore/pricing/).

## Citations are a condition of use

The Web Search terms require that the source citations and links that come with each
result are kept and shown in any output that reaches an end user. Step 6 shows how
to check that in code instead of relying on the prompt alone.

## Install the packages and set up the model

In [ ]:
%pip install -q -U langgraph langchain "langchain-aws[tools]>=1.8.0"

In [2]:
import json
import os
import re
import time
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from urllib.parse import urlparse

import boto3
from bedrock_agentcore.gateway.client import GatewayClient
from bedrock_agentcore.tools import WebSearchClient, WebSearchError
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver

from langchain_aws.tools import create_web_search_toolkit

#: The three regions that offer the web search connector.
WEB_SEARCH_REGIONS = ("us-east-1", "eu-west-1", "ap-northeast-1")

# Set deliberately rather than read from AWS_REGION. An ambient region such as
# us-west-2 creates a gateway happily and then fails the web search target with
# a message that reads like a missing entitlement.
REGION = "us-east-1"
if REGION not in WEB_SEARCH_REGIONS:
    raise ValueError(f"web search is offered in {', '.join(WEB_SEARCH_REGIONS)}")

# Cross-region inference profile. Any Bedrock chat model with tool use works.
MODEL_ID = "bedrock_converse:us.anthropic.claude-sonnet-5"
model = init_chat_model(MODEL_ID, region_name=REGION)


def answer_text(message):
    # A reasoning model answers with a list of content blocks, so printing
    # message.content directly would include the reasoning blocks too.
    if isinstance(message.content, str):
        return message.content
    return "\n".join(
        block["text"]
        for block in message.content
        if isinstance(block, dict) and block.get("type") == "text"
    )


def tool_calls(result):
    # The tool calls a run made, in order, as (name, args) pairs.
    return [
        (call["name"], call["args"])
        for message in result["messages"]
        for call in (getattr(message, "tool_calls", None) or [])
    ]


def domain_of(url):
    host = urlparse(url).hostname or ""
    return host[4:] if host.startswith("www.") else host


def matches(url, domains):
    # True when the URL's host is one of the domains or a subdomain of one.
    host = domain_of(url)
    return any(host == d or host.endswith("." + d) for d in domains)


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

## Step 1: Create a gateway with an open and a governed web search target

Web search reaches the service as an AgentCore Gateway connector target. This notebook
puts two targets on one gateway, because the difference between them is the point of
Step 4:

- **`web-search`** is the open web, filtered only by what each call asks for.
- **`vetted-sources`** carries an administrator blocklist of community sites
  (`wikipedia.org`, `reddit.com`, `quora.com`, including their subdomains). It's
  applied server-side to every call through this target, and it's hidden from the agent.

Both are pinned to connector version `1.2.0`, the first version with request-level
filters. Pinning makes the notebook independent of
whatever the connector's default happens to be when you run it.

Two IAM permissions are involved, on two different principals:

- The **gateway's service role** needs
  `bedrock-agentcore:InvokeWebSearch` on
  `arn:aws:bedrock-agentcore:<region>:aws:tool/web-search.v1`, and its trust policy
  must let `bedrock-agentcore.amazonaws.com` assume it. If the trust policy scopes
  `aws:SourceArn`, the region in it has to be the gateway's region. A mismatch is not
  rejected when the gateway is created; it shows up on the first search as
  `Failed to obtain execution role credentials`.
- **Your own credentials**, which the toolkit signs with, need
  `bedrock-agentcore:InvokeGateway` on the gateway. Creating the gateway with
  `AWS_IAM` inbound auth is what makes that grant the gate.

The first cell creates the service role if it doesn't exist yet, or uses the role in
`GATEWAY_ROLE_ARN` if you set one. The second cell reuses the gateway and targets if they
exist, and fails loudly if a reused target is pinned to an older connector version.

In [3]:
GATEWAY_ROLE_NAME = "AgentCoreWebSearchGatewayRole"
iam = boto3.client("iam")
account_id = boto3.client("sts").get_caller_identity()["Account"]

if os.environ.get("GATEWAY_ROLE_ARN"):
    GATEWAY_ROLE_ARN = os.environ["GATEWAY_ROLE_ARN"]
else:
    try:
        GATEWAY_ROLE_ARN = iam.get_role(RoleName=GATEWAY_ROLE_NAME)["Role"]["Arn"]
    except iam.exceptions.NoSuchEntityException:
        trust = {
            "Version": "2012-10-17",
            "Statement": [{
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {"aws:SourceAccount": account_id},
                    "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/*"},
                },
            }],
        }
        GATEWAY_ROLE_ARN = iam.create_role(
            RoleName=GATEWAY_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust),
            Description="Lets AgentCore Gateway call AgentCore Web Search",
        )["Role"]["Arn"]
        iam.put_role_policy(
            RoleName=GATEWAY_ROLE_NAME, PolicyName="InvokeWebSearch",
            PolicyDocument=json.dumps({
                "Version": "2012-10-17",
                "Statement": [{
                    "Effect": "Allow",
                    "Action": "bedrock-agentcore:InvokeWebSearch",
                    "Resource": f"arn:aws:bedrock-agentcore:{REGION}:aws:tool/web-search.v1",
                }],
            }),
        )
        time.sleep(10)  # let the new role propagate before the gateway assumes it
print(f"gateway service role: {GATEWAY_ROLE_ARN.split('/')[-1]}")

gateway service role: AgentCoreWebSearchGatewayRole


In [4]:
GATEWAY_NAME = "langchain-web-search-walkthrough"
OPEN_TARGET = "web-search"
GOVERNED_TARGET = "vetted-sources"
ADMIN_BLOCKLIST = ["wikipedia.org", "reddit.com", "quora.com"]
CONNECTOR_VERSION = "1.2.0"

gateway_client = GatewayClient(region_name=REGION)

gateway = gateway_client.get_gateway_by_name(GATEWAY_NAME)
if gateway is None:
    gateway = gateway_client.create_gateway_and_wait(
        name=GATEWAY_NAME,
        roleArn=GATEWAY_ROLE_ARN,
        authorizerType="AWS_IAM",  # callers sign with SigV4
        protocolType="MCP",
    )
GATEWAY_ID = gateway["gatewayId"]


def ensure_target(name, exclude_domains=None):
    target = gateway_client.get_gateway_target_by_name(GATEWAY_ID, name)
    if target is None:
        target = gateway_client.create_web_search_target(
            gateway_identifier=GATEWAY_ID,
            name=name,
            exclude_domains=exclude_domains,
            connector_version=CONNECTOR_VERSION,
        )
    version = target["targetConfiguration"]["mcp"]["connector"]["source"].get("version")
    if version != CONNECTOR_VERSION:
        raise RuntimeError(
            f"target {name} is on connector {version}; update it to {CONNECTOR_VERSION}"
        )
    admin = target["targetConfiguration"]["mcp"]["connector"]["configurations"][0]
    print(f"{name}: {target['status']}, connector {version}, admin policy {admin['parameterValues']}")


ensure_target(OPEN_TARGET)
ensure_target(GOVERNED_TARGET, exclude_domains=ADMIN_BLOCKLIST)

web-search: READY, connector 1.2.0, admin policy {}


vetted-sources: READY, connector 1.2.0, admin policy {'domainFilter': {'exclude': ['wikipedia.org', 'reddit.com', 'quora.com']}}


## Step 2: Create the toolkit and run a first search

`create_web_search_toolkit` returns one LangChain tool, `web_search`. Passing
`target_name` lets the client compute the gateway tool name
(`<target>___WebSearch`) instead of spending a `tools/list` round trip on it.

The toolkit signs with your ambient AWS credentials. There is nothing else to
configure: no key, no environment variable, no third-party account.

In [5]:
toolkit, tools = create_web_search_toolkit(
    region=REGION, gateway_id=GATEWAY_ID, target_name=OPEN_TARGET
)
web_search = tools[0]

print(f"tool name: {web_search.name}")
print(f"arguments the model can set: {list(web_search.args_schema.model_fields)}")
print()
print(web_search.invoke({"query": "Amazon Bedrock AgentCore web search", "max_results": 3}))

tool name: web_search
arguments the model can set: ['query', 'max_results', 'include_domains', 'exclude_domains', 'published_after', 'published_before']



1. Amazon Bedrock AgentCore
   URL: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-connector-web-search-tool.html
   Published: 05:00PM, Sunday, October 06 2024, PDT
   Web Search Tool on Amazon Bedrock AgentCore eliminates that complexity. It is a fully managed, MCP-compliant web search capability that lets your agents retrieve information from the web without any infrastructure overhead. It is available as a managed connector that you connect to your AgentCore Gateway. Agents discover it with a standard tools/list call and invoke it like any other Model Context Protocol tool. There are no search APIs to provision, no outbound credentials to manage, and no result-parsing glue to maintain. 

- Semantic snippet extraction — Pulls semantically relevant passages from each web page, optimized for a model’s context window, rather than returning raw HTML or full pages. Purpose-built web index Many solutions that add web search to an agent are wrappers over a thi

### Look at structured fact results

Some results aren't page extracts but structured facts: a small table with fields such as
`subject_name`, `section_title` and `Answer`, still carrying the source URL. They come
from the service's knowledge graph and tend to appear for entity and "at a glance"
questions. Whether a query returns any is up to the service, so this cell counts what
came back for a few queries and prints one if there is one.

In [6]:
sdk_client = WebSearchClient(region=REGION, gateway_id=GATEWAY_ID, target_name=OPEN_TARGET)


def is_structured(result):
    return result.text.lstrip().startswith("| subject_name")


example = None
for question in ["AWS re:Invent 2026 dates and venues", "Who is the CEO of Amazon?", "Amazon Bedrock AgentCore web search"]:
    response = sdk_client.search(question, max_results=10)
    structured = [r for r in response.results if is_structured(r)]
    print(f"{question!r}: {len(response.results)} results, {len(structured)} structured")
    example = example or (structured[0] if structured else None)

if example:
    print(f"\nexample structured result, source {example.url}:\n{example.text[:700]}")

'AWS re:Invent 2026 dates and venues': 10 results, 2 structured


'Who is the CEO of Amazon?': 10 results, 2 structured


'Amazon Bedrock AgentCore web search': 10 results, 2 structured

example structured result, source https://vendelux.com/lead-retrieval/aws-reinvent-2026:
| subject_name | AWS re:Invent 2026 |
| subject_type | event |
| section_title | AWS re:Invent 2026 at a glance |
| Answer | Detail Information Dates Monday, November 30 to Friday, December 4, 2026 Venues Caesars Forum, Caesars Palace, Encore, MGM Grand, The Venetian, and Wynn, Las Vegas Expo The Venetian: Tuesday, December 1 and Wednesday, December 2 from 10am to 6pm; Thursday, December 3 from 10am to 4pm Pass Full conference pass required ($2,499 full price); no day or Expo-only passes Sponsors 450+ AWS Partners in the Expo Lead retrieval Devices provided through RainFocus; outside devices, software, and APIs not permitted for sponsors Age Attendees must be 18 or older when they pick up th


## Step 3: Build an agent that chooses its own filters

`create_agent` builds a LangGraph graph. The model decides when to search and what
filters to pass; every filter is an ordinary tool argument it can see in the schema.
The helper prints the calls so you can see exactly what the model asked for.

In [7]:
SYSTEM_PROMPT = '''You are a research assistant with access to web search.

Search whenever a question touches recent events, releases, prices or anything that
may have changed since your training data. Every fact taken from a search result
must carry its source URL. End with the list of URLs you used. If the results do not
answer the question, say so.'''

agent = create_agent(model, tools=tools, system_prompt=SYSTEM_PROMPT)

result = agent.invoke(
    {"messages": [{"role": "user", "content":
        "Using only docs.aws.amazon.com, what is the maximum query length for the "
        "AgentCore web search tool, and how many results can one call return?"}]}
)
for name, args in tool_calls(result):
    print(f"tool call -> {name}({json.dumps(args)})")
print()
print(answer_text(result["messages"][-1]))

tool call -> web_search({"query": "AgentCore web search tool maximum query length results", "include_domains": ["docs.aws.amazon.com"]})

According to the AWS documentation for the AgentCore **Web Search Tool** (gateway target connector), per docs.aws.amazon.com:

- **Maximum query length:** The `query` string must be **200 characters or fewer**.
- **Maximum results per call:** The `maxResults` parameter has a **valid range of 1–25**, with a **default of 10** results.

Source: *"query | string | Yes | The search query string. Must be 200 characters or fewer. maxResults | integer | No | Maximum number of results to return. Valid range: 1-25. Defaults to 10."*

**URL used:** https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-connector-web-search-tool.html


### Compare a search with and without a date filter

`published_after` and `published_before` filter on the publication date the index
reports, inclusive, in ISO-8601 UTC. Results without a recognized date are dropped
when the filter is on, so a narrow window can return fewer results. The same query
with and without a 14-day window:

In [8]:
QUERY = "Amazon Bedrock AgentCore announcement"
since = datetime.now(timezone.utc) - timedelta(days=14)

unfiltered = sdk_client.search(QUERY, max_results=10)
recent = sdk_client.search(QUERY, max_results=10, published_after=iso(since))


def summarize(label, response):
    dates = [r.published_date or "no date" for r in response.results]
    print(f"{label}: {len(response.results)} results")
    for d in dates:
        print(f"   {d}")


summarize("no filter", unfiltered)
summarize(f"published after {iso(since)}", recent)

no filter: 10 results
   08:31AM, Friday, September 18 2026, PDT
   05:00PM, Thursday, September 17 2026, PDT
   05:00PM, Tuesday, June 16 2026, PDT
   05:00PM, Wednesday, August 05 2026, PDT
   05:00PM, Wednesday, June 17 2026, PDT
   05:00PM, Tuesday, June 16 2026, PDT
   11:56AM, Tuesday, August 18 2026, PDT
   12:13PM, Wednesday, April 22 2026, PDT
   12:04PM, Monday, October 12 2026, PDT
   12:04PM, Monday, October 12 2026, PDT
published after 2026-09-28T19:04:53Z: 10 results
   05:00PM, Tuesday, September 29 2026, PDT
   09:00AM, Tuesday, October 06 2026, PDT
   01:23PM, Wednesday, October 07 2026, PDT
   02:15PM, Thursday, October 08 2026, PDT
   08:39AM, Monday, October 12 2026, PDT
   07:34AM, Monday, October 12 2026, PDT
   01:45PM, Wednesday, October 07 2026, PDT
   01:45PM, Wednesday, October 07 2026, PDT
   11:33AM, Thursday, October 08 2026, PDT
   10:39AM, Monday, October 12 2026, PDT


## Step 4: Enforce a source blocklist on the gateway target

Step 3's filters are chosen by the model, which makes them a convenience rather
than a control: a different prompt, or a prompt injection in a page the agent reads,
can change them. For a rule that has to hold, put it on the gateway target.

The `vetted-sources` target carries an admin blocklist. Request filters compose with it
and can't lift it: a domain is dropped if it's on the admin list or on the caller's own
exclude list. This cell checks that against live results:

- the open target returns blocklisted domains for an ordinary query;
- the governed target never does, for the same query;
- a call that explicitly asks for a blocklisted domain still gets none of it.

In [9]:
governed_toolkit, governed_tools = create_web_search_toolkit(
    region=REGION, gateway_id=GATEWAY_ID, target_name=GOVERNED_TARGET
)
governed_client = WebSearchClient(region=REGION, gateway_id=GATEWAY_ID, target_name=GOVERNED_TARGET)

QUERY = "history of the Python programming language"


def blocked(response):
    return [r.url for r in response.results if r.url and matches(r.url, ADMIN_BLOCKLIST)]


open_results = sdk_client.search(QUERY, max_results=10)
print(f"open target:     {len(open_results.results)} results, {len(blocked(open_results))} from {ADMIN_BLOCKLIST}")
for url in blocked(open_results)[:3]:
    print(f"   {url}")

governed_results = governed_client.search(QUERY, max_results=10)
print(f"governed target: {len(governed_results.results)} results, {len(blocked(governed_results))} from the blocklist")

try:
    asked = governed_client.search(QUERY, max_results=10, include_domains=["wikipedia.org"])
    print(f"governed target, caller asks for wikipedia.org: {len(asked.results)} results, "
          f"{len(blocked(asked))} from the blocklist")
except WebSearchError as error:
    print(f"governed target, caller asks for wikipedia.org: refused\n   {error}")

open target:     10 results, 4 from ['wikipedia.org', 'reddit.com', 'quora.com']
   https://en.wikipedia.org/wiki/History_of_Python
   https://en.wikipedia.org/wiki/Python_(programming_language)
   https://en.wikipedia.org/wiki/Python_(programming_language)


governed target: 10 results, 0 from the blocklist


governed target, caller asks for wikipedia.org: 0 results, 0 from the blocklist


The same holds when the model drives. Asked for a source the policy blocks, the agent can
try, but the target never returns it. Setting `handle_tool_error` lets the model read any
error and carry on, instead of the run stopping.

In [10]:
for governed_tool in governed_tools:
    governed_tool.handle_tool_error = True

governed_agent = create_agent(model, tools=governed_tools, system_prompt=SYSTEM_PROMPT)
result = governed_agent.invoke(
    {"messages": [{"role": "user", "content":
        "Look up the history of the Python programming language on Wikipedia and summarize it."}]}
)
for name, args in tool_calls(result):
    print(f"tool call -> {name}({json.dumps(args)})")
searched = [
    url
    for message in result["messages"] if message.type == "tool"
    for url in re.findall(r"URL: (\S+)", message.content)
]
print(f"\nURLs the tool returned: {len(searched)}, from the blocklist: {sum(matches(u, ADMIN_BLOCKLIST) for u in searched)}")
print()
print(answer_text(result["messages"][-1])[:1500])

tool call -> web_search({"query": "Python programming language history Wikipedia"})

URLs the tool returned: 10, from the blocklist: 0

## Summary: History of the Python Programming Language

**Origins (late 1980s–1991)**
Python was conceived in the late 1980s by Guido van Rossum at Centrum Wiskunde & Informatica (CWI) in the Netherlands. It was designed as a successor to the ABC programming language (itself inspired by SETL), intended to support exception handling and interface with the Amoeba operating system. Implementation began in December 1989. Van Rossum first published the code publicly (version 0.9.1) to the alt.sources newsgroup in February 1991. Even at this early stage, the language already featured classes with inheritance, exception handling, functions, and core datatypes like list, dict, and str. It borrowed a module system from Modula-3, and its exception model also resembled Modula-3's. The language was named after the British comedy show *Monty Python's Flying Circus*

## Step 5: Apply a per-tenant blocklist from runtime context

A multi-tenant application often needs a different source policy per customer, for
example a list of sites a customer's contract says must never be cited, without
a target per tenant and without trusting the model to pass the right filters. In
LangGraph, that policy belongs in the run's **runtime context**: it is supplied by your
code with each invocation, and a `ToolRuntime` parameter is never shown to the model.

The `web_search` tool from the toolkit exposes filters only as model-visible arguments,
so this section defines a small tool on top of the AgentCore SDK client. It reads the
tenant's policy from context, merges it into every call, and returns the results both
as text for the model and as structured data (an artifact) for your code.

In [11]:
@dataclass
class TenantContext:
    tenant_id: str
    exclude_domains: list


@tool(response_format="content_and_artifact")
def tenant_web_search(query: str, runtime: ToolRuntime[TenantContext]) -> tuple:
    '''Search the web for current information. Cite the URLs you use.'''
    policy = runtime.context
    response = sdk_client.search(
        query,
        max_results=8,
        exclude_domains=policy.exclude_domains or None,
    )
    sources = [
        {"title": r.title, "url": r.url, "published": r.published_date}
        for r in response.results
    ]
    text = "\n\n".join(
        f"{i}. {r.title or 'Knowledge graph fact'}\n"
        + (f"   URL: {r.url}\n" if r.url else "   (no URL: knowledge graph fact)\n")
        + f"   {r.text[:800]}"
        for i, r in enumerate(response.results, start=1)
    ) or "No results."
    return text, sources


print("arguments the model sees:", list(tenant_web_search.tool_call_schema.model_json_schema()["properties"]))

arguments the model sees: ['query']


In [12]:
TENANTS = {
    "no-wikipedia": TenantContext("no-wikipedia", ["wikipedia.org", "wikiwand.com"]),
    "no-britannica": TenantContext("no-britannica", ["britannica.com"]),
}

tenant_agent = create_agent(
    model,
    tools=[tenant_web_search],
    system_prompt=SYSTEM_PROMPT,
    context_schema=TenantContext,
)

QUESTION = "Who created Python and when was it first released? Search the web and cite sources."
for tenant in TENANTS.values():
    result = tenant_agent.invoke(
        {"messages": [{"role": "user", "content": QUESTION}]}, context=tenant
    )
    sources = [s for m in result["messages"] if m.type == "tool" for s in (m.artifact or [])]
    off_policy = [s["url"] for s in sources if s["url"] and matches(s["url"], tenant.exclude_domains)]
    print(f"=== {tenant.tenant_id} (blocked: {tenant.exclude_domains})")
    print(f"tool calls: {[args for _, args in tool_calls(result)]}")
    print(f"sources returned: {len(sources)}, from blocked domains: {len(off_policy)}")
    print(f"domains returned: {sorted({domain_of(s['url']) for s in sources if s['url']})}")
    print(answer_text(result["messages"][-1])[:1200])
    print()

=== no-wikipedia (blocked: ['wikipedia.org', 'wikiwand.com'])
tool calls: [{'query': 'Who created Python programming language and when was it first released'}]
sources returned: 8, from blocked domains: 0
domains returned: ['aws.amazon.com', 'britannica.com', 'computerhistory.org', 'cvw.cac.cornell.edu', 'cwi.nl', 'ebsco.com', 'en.wikibooks.org', 'pythoninstitute.org']
**Python was created by Guido van Rossum**, a Dutch programmer, while he was working at the Centrum Wiskunde & Informatica (CWI) research institute in Amsterdam.

**Timeline:**
- **December 1989**: Van Rossum began working on Python as a hobby project during his Christmas vacation, intending it as a successor to the ABC programming language ([CWI](https://www.cwi.nl/en/news/25-years-of-python-at-cwi/); [AWS](https://aws.amazon.com/what-is/python/)).
- **February 20, 1991**: The first public release, **Python 0.9.0**, was made available ([Python Institute](https://pythoninstitute.org/about-python); [Wikibooks](https://en.

=== no-britannica (blocked: ['britannica.com'])
tool calls: [{'query': 'who created Python programming language and when was it first released'}]
sources returned: 8, from blocked domains: 0
domains returned: ['aws.amazon.com', 'en.wikipedia.org', 'pythoninstitute.org', 'wikiwand.com']
Python was created by **Guido van Rossum**. He began developing it in the late 1980s, starting implementation in December 1989 while working at Centrum Wiskunde & Informatica (CWI) in the Netherlands, as a successor to the ABC programming language. The first official public release, version **0.9.0**, came out on **February 20, 1991**. The language's name was inspired by the British comedy series *Monty Python's Flying Circus*.

Sources:
- https://en.wikipedia.org/wiki/History_of_Python
- https://en.wikipedia.org/wiki/Python_(programming_language)
- https://pythoninstitute.org/about-python
- https://aws.amazon.com/what-is/python/



## Step 6: Check citations in code

Because `tenant_web_search` returns its results as an artifact, each `ToolMessage`
carries the exact list of sources the model was shown. That makes the citation
requirement checkable in code: compare the URLs in the answer with the URLs the
searches returned. A URL in the answer that no search returned did not come from
web search, and an application can block or flag the answer before a user sees it.

In [13]:
def check_citations(result):
    returned = {
        s["url"].rstrip("/")
        for m in result["messages"] if m.type == "tool"
        for s in (m.artifact or []) if s.get("url")
    }
    answer = answer_text(result["messages"][-1])
    cited = {u.rstrip("/).,") for u in re.findall(r"https?://[^\s)\]>]+", answer)}
    return {
        "urls returned by search": len(returned),
        "urls cited in answer": len(cited),
        "cited and returned": sorted(cited & returned),
        "cited but never returned": sorted(cited - returned),
    }


result = tenant_agent.invoke(
    {"messages": [{"role": "user", "content":
        "What new features did the most recent Python release add? Cite sources."}]},
    context=TENANTS["no-wikipedia"],
)
print(json.dumps(check_citations(result), indent=2))

{
  "urls returned by search": 34,
  "urls cited in answer": 10,
  "cited and returned": [
    "https://blog.python.org/2026/10/python-3150-rc3",
    "https://docs.python.org/3.16/whatsnew/3.15.html",
    "https://docs.python.org/id/3.16/whatsnew/3.15.html",
    "https://realpython.com/python-news-june-2026",
    "https://realpython.com/python315-new-features",
    "https://www.bacancytechnology.com/blog/python-3-15",
    "https://www.heise.de/en/news/Python-3-15-released-Lazy-imports-UTF-8-and-new-data-types-11484310.html",
    "https://www.infoworld.com/article/4166693/the-best-new-features-in-python-3-15.html",
    "https://www.linuxcompatible.org/story/python-3150-is-out-lazy-imports-frozendict-and-faster-speeds",
    "https://www.python.org/downloads/release/python-3150a7"
  ],
  "cited but never returned": []
}


## Step 7: Handle search errors and multi-turn conversations

### Let the model correct a failed search

A failed search raises `ToolException`, so retry wrappers and callbacks record it as a
failure. To let the model read the error and fix its own call instead, set
`handle_tool_error`. The query limit is 200 characters, checked client-side before any
request is sent:

In [14]:
web_search.handle_tool_error = True
print(web_search.invoke({"query": "agentcore " * 30}))

Web search failed: query must be 200 characters or fewer, got 300


Web search failed: query must be 200 characters or fewer, got 300


### Continue a conversation across turns

A checkpointer keeps the conversation, including earlier search results, across turns
of the same thread. `InMemorySaver` is used here; for durable threads on AWS,
`langgraph-checkpoint-aws` provides `AgentCoreMemorySaver`, backed by AgentCore Memory.

In [15]:
threaded_agent = create_agent(
    model, tools=tools, system_prompt=SYSTEM_PROMPT, checkpointer=InMemorySaver()
)
thread = {"configurable": {"thread_id": "walkthrough-1"}}

first = threaded_agent.invoke(
    {"messages": [{"role": "user", "content": "In which AWS Regions is AgentCore Web Search available?"}]},
    thread,
)
print(answer_text(first["messages"][-1])[:800])
print("\n--- follow-up ---\n")
second = threaded_agent.invoke(
    {"messages": [{"role": "user", "content": "Which of those is in Europe?"}]}, thread
)
print(f"searches on the follow-up turn: {len(tool_calls(second)) - len(tool_calls(first))}")
print(answer_text(second["messages"][-1])[:800])

Based on current AWS documentation, **Web Search on Amazon Bedrock AgentCore** is available in three AWS Regions:

1. **US East (N. Virginia)** — `us-east-1` (original GA region, launched June 2026)
2. **Europe (Ireland)** — `eu-west-1` (added August 2026)
3. **Asia Pacific (Tokyo)** — `ap-northeast-1` (added August 2026)

Note that domain filtering and published-date filtering features for Web Search queries are available in these same three regions as well.

This matches the official AWS "Supported AWS Regions" table for AgentCore, which lists the "Web Search Tool" as supported (✓ Yes) only in US East (N. Virginia), Europe (Ireland), and Asia Pacific (Tokyo), with "No" for all other AgentCore-supported regions.

Sources:
- https://aws.amazon.com/about-aws/whats-new/2026/08/web-search-ama

--- follow-up ---



searches on the follow-up turn: 0
Of the three AWS Regions where AgentCore Web Search is available, the one in Europe is:

- **Europe (Ireland)** — region code `eu-west-1`

(The other two — US East (N. Virginia) `us-east-1` and Asia Pacific (Tokyo) `ap-northeast-1` — are in North America and Asia, respectively.)

Sources:
- https://aws.amazon.com/about-aws/whats-new/2026/08/web-search-amazon-bedrock/
- https://aws.amazon.com/blogs/machine-learning/domain-and-publish-date-filters-for-web-search-on-agentcore/


## Use the same gateway from other MCP clients

Nothing in this notebook is specific to LangChain on the gateway side. The gateway is an
MCP endpoint, so the same targets, including the admin blocklist from Step 4, serve
any MCP client that can authenticate to it. Claude Code is one example: it connects
through the `mcp-proxy-for-aws` stdio proxy, which signs requests with your AWS profile.

```bash
claude mcp add agentcore-web-search --scope user -- \
  uvx mcp-proxy-for-aws-cli@latest <gateway-url> --service bedrock-agentcore --region us-east-1
```

One gateway can then back a LangGraph service in production and developers' coding
assistants, with one policy and one bill.

## Clean up

`close()` releases the HTTP connections the clients hold. The gateway and its targets
are left in place unless you set `DELETE_GATEWAY = True`. The IAM role is left in
place either way, so other gateways can use it; delete it in the IAM console when you no
longer need it.

In [16]:
toolkit.close()
governed_toolkit.close()
sdk_client.close()
governed_client.close()

DELETE_GATEWAY = False

if DELETE_GATEWAY:
    for name in (OPEN_TARGET, GOVERNED_TARGET):
        target = gateway_client.get_gateway_target_by_name(GATEWAY_ID, name)
        if target:
            gateway_client.delete_gateway_target_and_wait(
                gatewayIdentifier=GATEWAY_ID, targetId=target["targetId"]
            )
    gateway_client.delete_gateway_and_wait(gatewayIdentifier=GATEWAY_ID)
    print(f"deleted {GATEWAY_ID}")
else:
    print("clients closed, gateway left in place")

clients closed, gateway left in place
